In [3]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv("../.env")

engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

with engine.connect() as conn:
    print(conn.execute(text("SELECT 1")).fetchone())

(1,)


In [4]:
import os
from dotenv import load_dotenv

load_dotenv("../.env", override=True)

API_KEY = os.getenv("MOLIT_API_KEY")

print("API KEY 로드:", API_KEY is not None)

API KEY 로드: True


In [11]:
import xml.etree.ElementTree as ET
import pandas as pd

root = ET.fromstring(response.text)

rows = []

for item in root.findall(".//item"):
    rows.append({
        child.tag: child.text
        for child in item
    })

df = pd.DataFrame(rows)

print(df.shape)
display(df.head())

(10, 25)


,aptNm,aptSeq,buildYear,contractTerm,contractType,dealDay,dealMonth,dealYear,deposit,excluUseAr,...,roadnm,roadnmbcd,roadnmbonbun,roadnmbubun,roadnmcd,roadnmseq,roadnmsggcd,sggCd,umdNm,useRRRight
0,은마,11680-218,1979,,,18,1,2024,"20,000",84.43,...,삼성로 212,0,00212,00000,3122005,4,11680,11680,대치동,
1,대치SKVIEW,11680-4637,2017,,,22,1,2024,"140,000",112.3685,...,삼성로51길 25,0,00025,00000,4166413,1,11680,11680,대치동,
2,한보미도맨션1,11680-222,1983,,,11,1,2024,"70,000",128.01,...,삼성로 150,0,00150,00000,3122005,4,11680,11680,대치동,
3,신동아,11680-319,1992,,,10,1,2024,"33,000",39.53,...,광평로47길 17,0,00017,00000,4166080,1,11680,11680,수서동,
4,수서한아름,11680-318,1993,,,27,1,2024,"110,000",163.68,...,광평로51길 22,0,00022,00000,4166081,1,11680,11680,수서동,


In [12]:
print(df.shape)
print(df.columns.tolist())

(10, 25)
['aptNm', 'aptSeq', 'buildYear', 'contractTerm', 'contractType', 'dealDay', 'dealMonth', 'dealYear', 'deposit', 'excluUseAr', 'floor', 'jibun', 'monthlyRent', 'preDeposit', 'preMonthlyRent', 'roadnm', 'roadnmbcd', 'roadnmbonbun', 'roadnmbubun', 'roadnmcd', 'roadnmseq', 'roadnmsggcd', 'sggCd', 'umdNm', 'useRRRight']


In [13]:
url = (
    "https://apis.data.go.kr/1613000/RTMSDataSvcAptRent/getRTMSDataSvcAptRent"
    f"?serviceKey={API_KEY}"
    "&LAWD_CD=11680"
    "&DEAL_YMD=202401"
    "&numOfRows=1000"
    "&pageNo=1"
)

response = requests.get(url)

print(response.status_code)

200


In [14]:
print(df.shape)

(10, 25)


In [15]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd

url = (
    "https://apis.data.go.kr/1613000/RTMSDataSvcAptRent/getRTMSDataSvcAptRent"
    f"?serviceKey={API_KEY}"
    "&LAWD_CD=11680"
    "&DEAL_YMD=202401"
    "&numOfRows=1000"
    "&pageNo=1"
)

response = requests.get(url)

root = ET.fromstring(response.text)

rows = []
for item in root.findall(".//item"):
    rows.append({child.tag: child.text for child in item})

df = pd.DataFrame(rows)

print(response.status_code)
print(df.shape)

200
(1000, 25)


In [16]:
total_count = root.findtext(".//totalCount")
print("전체 거래 건수:", total_count)

전체 거래 건수: 2464


In [17]:
url = (
    "https://apis.data.go.kr/1613000/RTMSDataSvcAptRent/getRTMSDataSvcAptRent"
    f"?serviceKey={API_KEY}"
    "&LAWD_CD=11680"
    "&DEAL_YMD=202401"
    "&numOfRows=3000"
    "&pageNo=1"
)

response = requests.get(url)

root = ET.fromstring(response.text)

rows = []
for item in root.findall(".//item"):
    rows.append({child.tag: child.text for child in item})

df = pd.DataFrame(rows)

print(df.shape)

(2464, 25)


In [18]:
df.to_csv(
    "../data/raw/gangnam_rent_202401.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")

저장 완료


In [19]:
df.isnull().sum()

aptNm               0
aptSeq              0
buildYear           0
contractTerm        0
contractType        0
dealDay             0
dealMonth           0
dealYear            0
deposit             0
excluUseAr          0
floor               0
jibun               0
monthlyRent         0
preDeposit          0
preMonthlyRent      0
roadnm              0
roadnmbcd         559
roadnmbonbun        0
roadnmbubun         0
roadnmcd            0
roadnmseq           0
roadnmsggcd         0
sggCd               0
umdNm               0
useRRRight          0
dtype: int64

In [20]:
df["deposit"] = (
    df["deposit"]
    .str.replace(",", "", regex=False)
    .astype(int)
)

df["monthlyRent"] = pd.to_numeric(df["monthlyRent"], errors="coerce")

print(df[["deposit", "monthlyRent"]].dtypes)
df[["deposit", "monthlyRent"]].head()

deposit        int64
monthlyRent    int64
dtype: object


,deposit,monthlyRent
0,20000,160
1,140000,0
2,70000,230
3,33000,0
4,110000,0


In [21]:
numeric_cols = [
    "excluUseAr",
    "floor",
    "buildYear",
    "dealYear",
    "dealMonth",
    "dealDay"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df[numeric_cols].dtypes)

excluUseAr    float64
floor           int64
buildYear       int64
dealYear        int64
dealMonth       int64
dealDay         int64
dtype: object


In [22]:
df["dealDate"] = pd.to_datetime(
    df["dealYear"].astype(str) + "-" +
    df["dealMonth"].astype(str) + "-" +
    df["dealDay"].astype(str)
)

print(df["dealDate"].head())
print(df["dealDate"].dtype)

0   2024-01-18
1   2024-01-22
2   2024-01-11
3   2024-01-10
4   2024-01-27
Name: dealDate, dtype: datetime64[us]
datetime64[us]


In [23]:
df["rentType"] = df["monthlyRent"].apply(
    lambda x: "전세" if x == 0 else "월세"
)

print(df["rentType"].value_counts())

rentType
전세    1435
월세    1029
Name: count, dtype: int64


In [24]:
df.to_csv(
    "../data/processed/gangnam_rent_202401_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print(df.shape)
print("정제 데이터 저장 완료")

(2464, 27)
정제 데이터 저장 완료


In [2]:
import pandas as pd

df = pd.read_csv(
    "../data/processed/gangnam_rent_202401_clean.csv"
)

print(df.shape)

(2464, 27)


In [3]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv("../.env", override=True)

engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

print("MySQL 연결 준비 완료")

MySQL 연결 준비 완료


In [4]:
df.to_sql(
    name="rental_transactions",
    con=engine,
    if_exists="replace",
    index=False
)

print("MySQL 저장 완료")

MySQL 저장 완료


In [5]:
from sqlalchemy import text

with engine.connect() as conn:
    result = conn.execute(
        text("SELECT COUNT(*) FROM rental_transactions")
    )
    print(result.fetchone())

(2464,)


In [6]:
query = """
SELECT
    rentType,
    COUNT(*) AS contract_count,
    ROUND(AVG(deposit), 0) AS avg_deposit,
    ROUND(AVG(monthlyRent), 0) AS avg_monthly_rent
FROM rental_transactions
GROUP BY rentType;
"""

result = pd.read_sql(query, engine)
result

,rentType,contract_count,avg_deposit,avg_monthly_rent
0,월세,1029,39061.0,188.0
1,전세,1435,84837.0,0.0


In [8]:
import os
from dotenv import load_dotenv

load_dotenv("../.env", override=True)

API_KEY = os.getenv("MOLIT_API_KEY")

print("API KEY 로드:", API_KEY is not None)

API KEY 로드: True


In [9]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import time

all_rows = []

for month in range(1, 13):
    ymd = f"2024{month:02d}"

    url = (
        "https://apis.data.go.kr/1613000/RTMSDataSvcAptRent/getRTMSDataSvcAptRent"
        f"?serviceKey={API_KEY}"
        "&LAWD_CD=11680"
        f"&DEAL_YMD={ymd}"
        "&numOfRows=5000"
        "&pageNo=1"
    )

    response = requests.get(url)
    root = ET.fromstring(response.text)

    items = root.findall(".//item")

    for item in items:
        all_rows.append({
            child.tag: child.text
            for child in item
        })

    print(ymd, len(items), "건")

    time.sleep(0.2)

df_2024 = pd.DataFrame(all_rows)

print("전체:", df_2024.shape)

202401 2464 건
202402 1985 건
202403 1976 건
202404 1651 건
202405 2539 건
202406 2073 건
202407 1648 건
202408 1613 건
202409 1341 건
202410 1744 건
202411 1696 건
202412 2098 건
전체: (22828, 25)


In [10]:
df_2024.to_csv(
    "../data/raw/gangnam_rent_2024.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", df_2024.shape)

저장 완료: (22828, 25)


In [11]:
df = df_2024.copy()

# 가격
df["deposit"] = df["deposit"].str.replace(",", "", regex=False).astype(int)
df["monthlyRent"] = pd.to_numeric(df["monthlyRent"], errors="coerce")

# 숫자형
numeric_cols = [
    "excluUseAr", "floor", "buildYear",
    "dealYear", "dealMonth", "dealDay"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 거래일자
df["dealDate"] = pd.to_datetime(
    df["dealYear"].astype(str) + "-" +
    df["dealMonth"].astype(str) + "-" +
    df["dealDay"].astype(str)
)

# 전세/월세
df["rentType"] = df["monthlyRent"].apply(
    lambda x: "전세" if x == 0 else "월세"
)

print(df.shape)
print(df["rentType"].value_counts())

(22828, 27)
rentType
전세    12116
월세    10712
Name: count, dtype: int64


In [12]:
df.to_csv(
    "../data/processed/gangnam_rent_2024_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", df.shape)

저장 완료: (22828, 27)


In [13]:
df.to_sql(
    name="rental_transactions",
    con=engine,
    if_exists="replace",
    index=False
)

print("MySQL 저장 완료:", len(df))

MySQL 저장 완료: 22828


In [14]:
query = """
SELECT
    dealMonth,
    COUNT(*) AS contract_count
FROM rental_transactions
GROUP BY dealMonth
ORDER BY dealMonth;
"""

monthly = pd.read_sql(query, engine)
monthly

,dealMonth,contract_count
0,1,2464
1,2,1985
2,3,1976
3,4,1651
4,5,2539
5,6,2073
6,7,1648
7,8,1613
8,9,1341
9,10,1744


In [15]:
seoul_gu = {
    "종로구": "11110",
    "중구": "11140",
    "용산구": "11170",
    "성동구": "11200",
    "광진구": "11215",
    "동대문구": "11230",
    "중랑구": "11260",
    "성북구": "11290",
    "강북구": "11305",
    "도봉구": "11320",
    "노원구": "11350",
    "은평구": "11380",
    "서대문구": "11410",
    "마포구": "11440",
    "양천구": "11470",
    "강서구": "11500",
    "구로구": "11530",
    "금천구": "11545",
    "영등포구": "11560",
    "동작구": "11590",
    "관악구": "11620",
    "서초구": "11650",
    "강남구": "11680",
    "송파구": "11710",
    "강동구": "11740"
}

print(len(seoul_gu))

25


In [16]:
all_rows = []

for gu_name, gu_code in seoul_gu.items():
    for month in range(1, 13):
        ymd = f"2024{month:02d}"

        url = (
            "https://apis.data.go.kr/1613000/RTMSDataSvcAptRent/getRTMSDataSvcAptRent"
            f"?serviceKey={API_KEY}"
            f"&LAWD_CD={gu_code}"
            f"&DEAL_YMD={ymd}"
            "&numOfRows=5000"
            "&pageNo=1"
        )

        response = requests.get(url)
        root = ET.fromstring(response.text)
        items = root.findall(".//item")

        for item in items:
            row = {child.tag: child.text for child in item}
            row["guName"] = gu_name
            all_rows.append(row)

        print(gu_name, ymd, len(items))

        time.sleep(0.2)

df_seoul_2024 = pd.DataFrame(all_rows)

print("전체:", df_seoul_2024.shape)

종로구 202401 234
종로구 202402 173
종로구 202403 185
종로구 202404 134
종로구 202405 167
종로구 202406 151
종로구 202407 160
종로구 202408 143
종로구 202409 120
종로구 202410 147
종로구 202411 162
종로구 202412 239
중구 202401 398
중구 202402 372
중구 202403 373
중구 202404 309
중구 202405 324
중구 202406 270
중구 202407 329
중구 202408 269
중구 202409 255
중구 202410 302
중구 202411 253
중구 202412 298
용산구 202401 563
용산구 202402 503
용산구 202403 553
용산구 202404 709
용산구 202405 631
용산구 202406 586
용산구 202407 391
용산구 202408 464
용산구 202409 377
용산구 202410 470
용산구 202411 724
용산구 202412 662
성동구 202401 966
성동구 202402 844
성동구 202403 951
성동구 202404 778
성동구 202405 761
성동구 202406 771
성동구 202407 757
성동구 202408 762
성동구 202409 655
성동구 202410 791
성동구 202411 731
성동구 202412 863
광진구 202401 420
광진구 202402 388
광진구 202403 410
광진구 202404 326
광진구 202405 337
광진구 202406 343
광진구 202407 338
광진구 202408 360
광진구 202409 310
광진구 202410 377
광진구 202411 372
광진구 202412 498
동대문구 202401 818
동대문구 202402 701
동대문구 202403 769
동대문구 202404 584
동대문구 202405 651
동대문구 202406 606
동대문구 202407 700


In [17]:
df_seoul_2024.to_csv(
    "../data/raw/seoul_rent_2024.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", df_seoul_2024.shape)

저장 완료: (252914, 26)


In [18]:
df = df_seoul_2024.copy()

# 가격 숫자형 변환
df["deposit"] = (
    df["deposit"]
    .str.replace(",", "", regex=False)
    .astype(int)
)

df["monthlyRent"] = pd.to_numeric(df["monthlyRent"], errors="coerce")

# 기타 숫자형
numeric_cols = [
    "excluUseAr", "floor", "buildYear",
    "dealYear", "dealMonth", "dealDay"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 거래일
df["dealDate"] = pd.to_datetime(
    df["dealYear"].astype(str) + "-" +
    df["dealMonth"].astype(str) + "-" +
    df["dealDay"].astype(str)
)

# 전세 / 월세
df["rentType"] = df["monthlyRent"].apply(
    lambda x: "전세" if x == 0 else "월세"
)

print(df.shape)
print(df["rentType"].value_counts())

(252914, 28)
rentType
전세    145224
월세    107690
Name: count, dtype: int64


In [19]:
df.to_csv(
    "../data/processed/seoul_rent_2024_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", df.shape)

저장 완료: (252914, 28)


In [20]:
df.to_sql(
    name="rental_transactions",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=5000
)

print("MySQL 저장 완료:", len(df))

MySQL 저장 완료: 252914


In [21]:
query = """
SELECT
    guName,
    COUNT(*) AS contract_count
FROM rental_transactions
GROUP BY guName
ORDER BY contract_count DESC;
"""

gu_count = pd.read_sql(query, engine)

print(gu_count)
print("구 개수:", len(gu_count))
print("전체:", gu_count["contract_count"].sum())

   guName  contract_count
0     강남구           22828
1     송파구           21309
2     노원구           18722
3     강동구           16706
4     강서구           15611
5     서초구           14381
6    영등포구           12494
7     마포구           11787
8     양천구           11295
9     구로구           10645
10    은평구           10058
11    성동구            9630
12    성북구            9136
13    동작구            9004
14   동대문구            8493
15   서대문구            7558
16    중랑구            7269
17    관악구            6813
18    용산구            6633
19    도봉구            6053
20    광진구            4479
21     중구            3752
22    강북구            3225
23    금천구            3018
24    종로구            2015
구 개수: 25
전체: 252914


In [22]:
kosis = pd.read_csv(
    "../data/raw/kosis_migration_2024.csv",
    encoding="utf-8"
)

print(kosis.shape)
print(kosis.columns.tolist())
kosis.head()

(26, 73)
['행정구역(시군구)별', '2024.01', '2024.01.1', '2024.01.2', '2024.01.3', '2024.01.4', '2024.01.5', '2024.02', '2024.02.1', '2024.02.2', '2024.02.3', '2024.02.4', '2024.02.5', '2024.03', '2024.03.1', '2024.03.2', '2024.03.3', '2024.03.4', '2024.03.5', '2024.04', '2024.04.1', '2024.04.2', '2024.04.3', '2024.04.4', '2024.04.5', '2024.05', '2024.05.1', '2024.05.2', '2024.05.3', '2024.05.4', '2024.05.5', '2024.06', '2024.06.1', '2024.06.2', '2024.06.3', '2024.06.4', '2024.06.5', '2024.07', '2024.07.1', '2024.07.2', '2024.07.3', '2024.07.4', '2024.07.5', '2024.08', '2024.08.1', '2024.08.2', '2024.08.3', '2024.08.4', '2024.08.5', '2024.09', '2024.09.1', '2024.09.2', '2024.09.3', '2024.09.4', '2024.09.5', '2024.10', '2024.10.1', '2024.10.2', '2024.10.3', '2024.10.4', '2024.10.5', '2024.11', '2024.11.1', '2024.11.2', '2024.11.3', '2024.11.4', '2024.11.5', '2024.12', '2024.12.1', '2024.12.2', '2024.12.3', '2024.12.4', '2024.12.5']


,행정구역(시군구)별,2024.01,2024.01.1,2024.01.2,2024.01.3,2024.01.4,2024.01.5,2024.02,2024.02.1,2024.02.2,...,2024.11.2,2024.11.3,2024.11.4,2024.11.5,2024.12,2024.12.1,2024.12.2,2024.12.3,2024.12.4,2024.12.5
0,행정구역(시군구)별,총전입 (명),총전출 (명),시도내이동-시군구간 전입 (명),시도내이동-시군구간 전출 (명),시도간전입 (명),시도간전출 (명),총전입 (명),총전출 (명),시도내이동-시군구간 전입 (명),...,시도내이동-시군구간 전입 (명),시도내이동-시군구간 전출 (명),시도간전입 (명),시도간전출 (명),총전입 (명),총전출 (명),시도내이동-시군구간 전입 (명),시도내이동-시군구간 전출 (명),시도간전입 (명),시도간전출 (명)
1,종로구,1725,1704,846,914,713,624,2356,2208,1082,...,631,769,428,414,1363,1581,695,842,490,561
2,중구,1587,1551,820,833,601,552,2050,1908,996,...,626,752,363,425,1315,1658,659,908,439,533
3,용산구,2638,3600,1079,1913,1019,1147,3067,3988,1307,...,891,1160,646,737,2222,2452,1029,1252,813,820
4,성동구,3147,3410,1519,1751,1178,1209,4202,4365,1886,...,1228,1635,692,932,2523,3151,1230,1737,830,951


In [23]:
kosis = pd.read_csv(
    "../data/raw/kosis_migration_2024.csv",
    encoding="utf-8",
    header=[0, 1]
)

print(kosis.shape)
kosis.head()

(25, 73)


행정구역(시군구)별 2024.01                                                        \
  행정구역(시군구)별 총전입 (명) 총전출 (명) 시도내이동-시군구간 전입 (명) 시도내이동-시군구간 전출 (명) 시도간전입 (명)   
0        종로구    1725    1704               846               914       713   
1         중구    1587    1551               820               833       601   
2        용산구    2638    3600              1079              1913      1019   
3        성동구    3147    3410              1519              1751      1178   
4        광진구    3903    4104              1471              1682      1623   

            2024.02                            ...           2024.11  \
  시도간전출 (명) 총전입 (명) 총전출 (명) 시도내이동-시군구간 전입 (명)  ... 시도내이동-시군구간 전입 (명)   
0       624    2356    2208              1082  ...               631   
1       552    2050    1908               996  ...               626   
2      1147    3067    3988              1307  ...               891   
3      1209    4202    4365              1886  ...              1228   
4      1613    4948    5038              1878  ...              1319   

                                        2024.12                            \
  시도내이동-시군구간 전출 (명) 시도간전입 (명) 시도간전출 (명) 총전입 (명) 총전출 (명) 시도내이동-시군구간 전입 (명)   
0               769       428       414    1363    1581               695   
1               752       363       425    1315    1658               659   
2              1160       646       737    2222    2452              1029   
3              1635       692       932    2523    3151              1230   
4              1477       977      1095    3424    4038              1425   

                                         
  시도내이동-시군구간 전출 (명) 시도간전입 (명) 시도간전출 (명)  
0               842       490       561  
1               908       439       533  
2              1252       813       820  
3              1737       830       951  
4              1894      1215      1360  

[5 rows x 73 columns]

In [24]:
rows = []

for _, row in kosis.iterrows():
    gu = row.iloc[0]

    for month in range(1, 13):
        ym = f"2024.{month:02d}"

        rows.append({
            "guName": gu,
            "yearMonth": ym.replace(".", "-"),
            "moveIn": row[(ym, "총전입 (명)")],
            "moveOut": row[(ym, "총전출 (명)")]
        })

migration = pd.DataFrame(rows)

print(migration.shape)
migration.head()

(300, 4)


,guName,yearMonth,moveIn,moveOut
0,종로구,2024-01,1725,1704
1,종로구,2024-02,2356,2208
2,종로구,2024-03,1861,1798
3,종로구,2024-04,1595,1625
4,종로구,2024-05,1409,1469


In [25]:
migration["moveIn"] = pd.to_numeric(migration["moveIn"], errors="coerce")
migration["moveOut"] = pd.to_numeric(migration["moveOut"], errors="coerce")

migration["netMove"] = migration["moveIn"] - migration["moveOut"]

print(migration.dtypes)
migration.head()

guName         str
yearMonth      str
moveIn       int64
moveOut      int64
netMove      int64
dtype: object


,guName,yearMonth,moveIn,moveOut,netMove
0,종로구,2024-01,1725,1704,21
1,종로구,2024-02,2356,2208,148
2,종로구,2024-03,1861,1798,63
3,종로구,2024-04,1595,1625,-30
4,종로구,2024-05,1409,1469,-60


In [26]:
migration.to_csv(
    "../data/processed/kosis_migration_2024_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", migration.shape)

저장 완료: (300, 5)


In [27]:
migration.to_sql(
    name="migration",
    con=engine,
    if_exists="replace",
    index=False
)

print("MySQL 저장 완료:", len(migration))

MySQL 저장 완료: 300


In [28]:
query = """
SELECT
    guName,
    CONCAT(dealYear, '-', LPAD(dealMonth, 2, '0')) AS yearMonth,
    COUNT(*) AS contractCount
FROM rental_transactions
GROUP BY guName, dealYear, dealMonth
ORDER BY guName, dealYear, dealMonth;
"""

rent_monthly = pd.read_sql(query, engine)

print(rent_monthly.shape)
rent_monthly.head()

(300, 3)


,guName,yearMonth,contractCount
0,강남구,2024-01,2464
1,강남구,2024-02,1985
2,강남구,2024-03,1976
3,강남구,2024-04,1651
4,강남구,2024-05,2539


In [29]:
query = """
SELECT
    r.guName,
    r.yearMonth,
    r.contractCount,
    m.moveIn,
    m.moveOut,
    m.netMove
FROM (
    SELECT
        guName,
        CONCAT(dealYear, '-', LPAD(dealMonth, 2, '0')) AS yearMonth,
        COUNT(*) AS contractCount
    FROM rental_transactions
    GROUP BY guName, dealYear, dealMonth
) r
JOIN migration m
    ON r.guName = m.guName
    AND r.yearMonth = m.yearMonth
ORDER BY r.guName, r.yearMonth;
"""

market = pd.read_sql(query, engine)

print(market.shape)
market.head()

(300, 6)


,guName,yearMonth,contractCount,moveIn,moveOut,netMove
0,강남구,2024-01,2464,13140,8937,4203
1,강남구,2024-02,1985,14454,10088,4366
2,강남구,2024-03,1976,7752,7059,693
3,강남구,2024-04,1651,6948,6624,324
4,강남구,2024-05,2539,6824,5592,1232


In [30]:
market.to_sql(
    name="monthly_rent_market",
    con=engine,
    if_exists="replace",
    index=False
)

print("저장 완료:", len(market))

저장 완료: 300


In [31]:
ecos = pd.read_csv(
    "../data/raw/ecos_base_rate_2024.csv",
    encoding="utf-8"
)

print(ecos.shape)
print(ecos.columns.tolist())
ecos.head()

(12, 16)
['통계표', '계정항목', '단위', '변환', '2024/01', '2024/02', '2024/03', '2024/04', '2024/05', '2024/06', '2024/07', '2024/08', '2024/09', '2024/10', '2024/11', '2024/12']


,통계표,계정항목,단위,변환,2024/01,2024/02,2024/03,2024/04,2024/05,2024/06,2024/07,2024/08,2024/09,2024/10,2024/11,2024/12
0,1.3.1. 한국은행 기준금리 및 여수신금리,한국은행 기준금리,연%,원자료,3.500,3.500,3.500,3.500,3.500,3.500,3.500,3.500,3.500,3.250,3.000,3.000
1,1.3.1. 한국은행 기준금리 및 여수신금리,정부대출금금리,연%,원자료,3.623,3.623,3.623,3.563,3.563,3.563,3.543,3.543,3.543,3.302,3.302,3.302
2,1.3.1. 한국은행 기준금리 및 여수신금리,총액한도대출금리,연%,원자료,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.3.1. 한국은행 기준금리 및 여수신금리,결제자금지원한도 대출금리,연%,원자료,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1.3.1. 한국은행 기준금리 및 여수신금리,무역금융지원 프로그램대출금리,연%,원자료,2.000,2.000,2.000,2.000,2.000,2.000,2.000,2.000,2.000,1.750,1.500,1.500


In [32]:
print(ecos["계정항목"].tolist())

['한국은행 기준금리', '정부대출금금리', '총액한도대출금리', '결제자금지원한도 대출금리', '무역금융지원 프로그램대출금리', '신용대출지원 프로그램대출금리', '영세자영업자지원 프로그램대출금리', '신성장·일자리지원 프로그램대출금리', '설비투자지원 프로그램대출금리', '지방중소기업지원 프로그램대출금리', '자금조정 대출금리', '자금조정 예금금리']


In [33]:
rate_row = ecos[ecos["계정항목"] == "한국은행 기준금리"].iloc[0]

rates = []

for month in range(1, 13):
    col = f"2024/{month:02d}"

    rates.append({
        "yearMonth": f"2024-{month:02d}",
        "baseRate": rate_row[col]
    })

interest_rate = pd.DataFrame(rates)

interest_rate["baseRate"] = pd.to_numeric(
    interest_rate["baseRate"],
    errors="coerce"
)

print(interest_rate.shape)
interest_rate


(12, 2)


,yearMonth,baseRate
0,2024-01,3.50
1,2024-02,3.50
2,2024-03,3.50
3,2024-04,3.50
4,2024-05,3.50
5,2024-06,3.50
6,2024-07,3.50
7,2024-08,3.50
8,2024-09,3.50
9,2024-10,3.25


In [34]:
interest_rate.to_sql(
    name="interest_rate",
    con=engine,
    if_exists="replace",
    index=False
)

print("MySQL 저장 완료:", len(interest_rate))

MySQL 저장 완료: 12


In [35]:
query = """
SELECT
    m.guName,
    m.yearMonth,
    m.contractCount,
    m.moveIn,
    m.moveOut,
    m.netMove,
    i.baseRate
FROM monthly_rent_market m
JOIN interest_rate i
    ON m.yearMonth = i.yearMonth
ORDER BY m.guName, m.yearMonth;
"""

market_final = pd.read_sql(query, engine)

print(market_final.shape)
market_final.head()

(300, 7)


,guName,yearMonth,contractCount,moveIn,moveOut,netMove,baseRate
0,강남구,2024-01,2464,13140,8937,4203,3.5
1,강남구,2024-02,1985,14454,10088,4366,3.5
2,강남구,2024-03,1976,7752,7059,693,3.5
3,강남구,2024-04,1651,6948,6624,324,3.5
4,강남구,2024-05,2539,6824,5592,1232,3.5


In [36]:
market_final.to_sql(
    name="monthly_rent_market_final",
    con=engine,
    if_exists="replace",
    index=False
)

print("최종 테이블 저장 완료:", len(market_final))

최종 테이블 저장 완료: 300


In [37]:
seoul_pop = pd.read_csv(
    "../data/raw/LOCAL_PEOPLE_GU_2024.csv",
    encoding="utf-8"
)

print(seoul_pop.shape)
print(seoul_pop.columns.tolist())
seoul_pop.head()

(219600, 33)
['stdr_de_id', 'tmzon_pd_se', 'adstrd_code_se', 'tot_lvpop_co', 'male_f0t9_lvpop_co', 'male_f10t14_lvpop_co', 'male_f15t19_lvpop_co', 'male_f20t24_lvpop_co', 'male_f25t29_lvpop_co', 'male_f30t34_lvpop_co', 'male_f35t39_lvpop_co', 'male_f40t44_lvpop_co', 'male_f45t49_lvpop_co', 'male_f50t54_lvpop_co', 'male_f55t59_lvpop_co', 'male_f60t64_lvpop_co', 'male_f65t69_lvpop_co', 'male_f70t74_lvpop_co', 'female_f0t9_lvpop_co', 'female_f10t14_lvpop_co', 'female_f15t19_lvpop_co', 'female_f20t24_lvpop_co', 'female_f25t29_lvpop_co', 'female_f30t34_lvpop_co', 'female_f35t39_lvpop_co', 'female_f40t44_lvpop_co', 'female_f45t49_lvpop_co', 'female_f50t54_lvpop_co', 'female_f55t59_lvpop_co', 'female_f60t64_lvpop_co', 'female_f65t69_lvpop_co', 'female_f70t74_lvpop_co', 'ldadng_dt']


,stdr_de_id,tmzon_pd_se,adstrd_code_se,tot_lvpop_co,male_f0t9_lvpop_co,male_f10t14_lvpop_co,male_f15t19_lvpop_co,male_f20t24_lvpop_co,male_f25t29_lvpop_co,male_f30t34_lvpop_co,...,female_f30t34_lvpop_co,female_f35t39_lvpop_co,female_f40t44_lvpop_co,female_f45t49_lvpop_co,female_f50t54_lvpop_co,female_f55t59_lvpop_co,female_f60t64_lvpop_co,female_f65t69_lvpop_co,female_f70t74_lvpop_co,ldadng_dt
0,20240101,0,11110,231940.9131,6080.9363,3260.2185,7037.7728,11720.3719,11453.5946,9398.7762,...,8270.8979,8262.8127,7381.2943,8933.9256,8204.0952,8077.3270,6773.4574,5040.6403,13601.2481,20240106080757
1,20240101,0,11140,182354.5506,5902.9352,2159.7405,3988.1295,8122.6534,9192.2022,8468.1934,...,8025.3593,8376.7658,6437.4373,6552.0269,5470.2929,5710.5258,4853.0913,3468.4969,9611.2688,20240106080757
2,20240101,0,11170,255885.5460,7345.2048,3639.3568,5057.8888,9234.4375,12070.7663,12527.3087,...,11832.5148,13239.7348,10843.0989,10513.5132,9354.6379,8665.0049,7487.8939,5751.9700,15265.4809,20240106080757
3,20240101,0,11200,302502.4326,10581.9842,4442.1591,6422.9086,9393.7985,12003.4389,11298.4993,...,12224.1674,15389.4293,12996.5818,12855.6322,11768.9656,11444.6560,9825.7494,7485.4514,18810.1096,20240106080757
4,20240101,0,11215,360329.9580,11123.9665,5904.0852,11819.2905,15020.4273,16807.9300,14211.2764,...,13959.7815,15187.1613,13256.5885,14758.9090,12738.2737,13649.9078,11219.6742,8644.5426,20246.3334,20240106080757


In [38]:
seoul_pop = pd.read_csv(
    "../data/raw/LOCAL_PEOPLE_GU_2024.csv",
    encoding="utf-8"
)

print(seoul_pop.shape)
print(seoul_pop.columns.tolist())
seoul_pop.head()

(219600, 33)
['stdr_de_id', 'tmzon_pd_se', 'adstrd_code_se', 'tot_lvpop_co', 'male_f0t9_lvpop_co', 'male_f10t14_lvpop_co', 'male_f15t19_lvpop_co', 'male_f20t24_lvpop_co', 'male_f25t29_lvpop_co', 'male_f30t34_lvpop_co', 'male_f35t39_lvpop_co', 'male_f40t44_lvpop_co', 'male_f45t49_lvpop_co', 'male_f50t54_lvpop_co', 'male_f55t59_lvpop_co', 'male_f60t64_lvpop_co', 'male_f65t69_lvpop_co', 'male_f70t74_lvpop_co', 'female_f0t9_lvpop_co', 'female_f10t14_lvpop_co', 'female_f15t19_lvpop_co', 'female_f20t24_lvpop_co', 'female_f25t29_lvpop_co', 'female_f30t34_lvpop_co', 'female_f35t39_lvpop_co', 'female_f40t44_lvpop_co', 'female_f45t49_lvpop_co', 'female_f50t54_lvpop_co', 'female_f55t59_lvpop_co', 'female_f60t64_lvpop_co', 'female_f65t69_lvpop_co', 'female_f70t74_lvpop_co', 'ldadng_dt']


,stdr_de_id,tmzon_pd_se,adstrd_code_se,tot_lvpop_co,male_f0t9_lvpop_co,male_f10t14_lvpop_co,male_f15t19_lvpop_co,male_f20t24_lvpop_co,male_f25t29_lvpop_co,male_f30t34_lvpop_co,...,female_f30t34_lvpop_co,female_f35t39_lvpop_co,female_f40t44_lvpop_co,female_f45t49_lvpop_co,female_f50t54_lvpop_co,female_f55t59_lvpop_co,female_f60t64_lvpop_co,female_f65t69_lvpop_co,female_f70t74_lvpop_co,ldadng_dt
0,20240101,0,11110,231940.9131,6080.9363,3260.2185,7037.7728,11720.3719,11453.5946,9398.7762,...,8270.8979,8262.8127,7381.2943,8933.9256,8204.0952,8077.3270,6773.4574,5040.6403,13601.2481,20240106080757
1,20240101,0,11140,182354.5506,5902.9352,2159.7405,3988.1295,8122.6534,9192.2022,8468.1934,...,8025.3593,8376.7658,6437.4373,6552.0269,5470.2929,5710.5258,4853.0913,3468.4969,9611.2688,20240106080757
2,20240101,0,11170,255885.5460,7345.2048,3639.3568,5057.8888,9234.4375,12070.7663,12527.3087,...,11832.5148,13239.7348,10843.0989,10513.5132,9354.6379,8665.0049,7487.8939,5751.9700,15265.4809,20240106080757
3,20240101,0,11200,302502.4326,10581.9842,4442.1591,6422.9086,9393.7985,12003.4389,11298.4993,...,12224.1674,15389.4293,12996.5818,12855.6322,11768.9656,11444.6560,9825.7494,7485.4514,18810.1096,20240106080757
4,20240101,0,11215,360329.9580,11123.9665,5904.0852,11819.2905,15020.4273,16807.9300,14211.2764,...,13959.7815,15187.1613,13256.5885,14758.9090,12738.2737,13649.9078,11219.6742,8644.5426,20246.3334,20240106080757


In [39]:
print(seoul_pop["stdr_de_id"].head())
print(seoul_pop["stdr_de_id"].dtype)

0    20240101
1    20240101
2    20240101
3    20240101
4    20240101
Name: stdr_de_id, dtype: int64
int64


In [40]:
seoul_pop["yearMonth"] = (
    seoul_pop["stdr_de_id"]
    .astype(str)
    .str[:6]
    .str[:4] + "-" +
    seoul_pop["stdr_de_id"]
    .astype(str)
    .str[4:6]
)

print(seoul_pop["yearMonth"].unique())

<StringArray>
['2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07',
 '2024-08', '2024-09', '2024-10', '2024-11', '2024-12']
Length: 12, dtype: str


In [41]:
print(seoul_pop["adstrd_code_se"].nunique())
print(sorted(seoul_pop["adstrd_code_se"].unique()))

25
[np.int64(11110), np.int64(11140), np.int64(11170), np.int64(11200), np.int64(11215), np.int64(11230), np.int64(11260), np.int64(11290), np.int64(11305), np.int64(11320), np.int64(11350), np.int64(11380), np.int64(11410), np.int64(11440), np.int64(11470), np.int64(11500), np.int64(11530), np.int64(11545), np.int64(11560), np.int64(11590), np.int64(11620), np.int64(11650), np.int64(11680), np.int64(11710), np.int64(11740)]


In [42]:
code_to_gu = {int(code): name for name, code in seoul_gu.items()}

seoul_pop["guName"] = seoul_pop["adstrd_code_se"].map(code_to_gu)

print(seoul_pop["guName"].nunique())
print(seoul_pop["guName"].isnull().sum())

25
0


In [43]:
living_pop = (
    seoul_pop
    .groupby(["guName", "yearMonth"])["tot_lvpop_co"]
    .mean()
    .reset_index()
    .rename(columns={"tot_lvpop_co": "avgLivingPop"})
)

print(living_pop.shape)
living_pop.head()

(300, 3)


,guName,yearMonth,avgLivingPop
0,강남구,2024-01,814365.906419
1,강남구,2024-02,812009.586478
2,강남구,2024-03,826818.317374
3,강남구,2024-04,824744.923619
4,강남구,2024-05,816933.047707


In [44]:
living_pop.to_sql(
    name="living_population",
    con=engine,
    if_exists="replace",
    index=False
)

print("MySQL 저장 완료:", len(living_pop))

MySQL 저장 완료: 300


In [45]:
query = """
SELECT
    m.guName,
    m.yearMonth,
    m.contractCount,
    m.moveIn,
    m.moveOut,
    m.netMove,
    m.baseRate,
    l.avgLivingPop
FROM monthly_rent_market_final m
JOIN living_population l
    ON m.guName = l.guName
    AND m.yearMonth = l.yearMonth
ORDER BY m.guName, m.yearMonth;
"""

ml_data = pd.read_sql(query, engine)

print(ml_data.shape)
ml_data.head()

(300, 8)


,guName,yearMonth,contractCount,moveIn,moveOut,netMove,baseRate,avgLivingPop
0,강남구,2024-01,2464,13140,8937,4203,3.5,814365.906419
1,강남구,2024-02,1985,14454,10088,4366,3.5,812009.586478
2,강남구,2024-03,1976,7752,7059,693,3.5,826818.317374
3,강남구,2024-04,1651,6948,6624,324,3.5,824744.923619
4,강남구,2024-05,2539,6824,5592,1232,3.5,816933.047707


In [46]:
ml_data.to_sql(
    name="ml_rent_market",
    con=engine,
    if_exists="replace",
    index=False
)

print("ML 테이블 저장 완료:", len(ml_data))

ML 테이블 저장 완료: 300


여기까지 2024년 이후 2023년, 2025년 추가

In [47]:
years = [2023, 2024, 2025]

months = [
    f"{year}{month:02d}"
    for year in years
    for month in range(1, 13)
]

print(months)
print("총 개월 수:", len(months))

['202301', '202302', '202303', '202304', '202305', '202306', '202307', '202308', '202309', '202310', '202311', '202312', '202401', '202402', '202403', '202404', '202405', '202406', '202407', '202408', '202409', '202410', '202411', '202412', '202501', '202502', '202503', '202504', '202505', '202506', '202507', '202508', '202509', '202510', '202511', '202512']
총 개월 수: 36


In [48]:
all_rows_extra = []

for gu_name, gu_code in seoul_gu.items():
    for year in [2023, 2025]:
        for month in range(1, 13):
            ymd = f"{year}{month:02d}"

            url = (
                "https://apis.data.go.kr/1613000/RTMSDataSvcAptRent/getRTMSDataSvcAptRent"
                f"?serviceKey={API_KEY}"
                f"&LAWD_CD={gu_code}"
                f"&DEAL_YMD={ymd}"
                "&numOfRows=5000"
                "&pageNo=1"
            )

            response = requests.get(url)
            root = ET.fromstring(response.text)
            items = root.findall(".//item")

            for item in items:
                row = {child.tag: child.text for child in item}
                row["guName"] = gu_name
                all_rows_extra.append(row)

            print(gu_name, ymd, len(items))
            time.sleep(0.2)

df_seoul_extra = pd.DataFrame(all_rows_extra)

print("추가 데이터:", df_seoul_extra.shape)

종로구 202301 180
종로구 202302 205
종로구 202303 193
종로구 202304 189
종로구 202305 211
종로구 202306 165
종로구 202307 182
종로구 202308 218
종로구 202309 219
종로구 202310 196
종로구 202311 236
종로구 202312 223
종로구 202501 184
종로구 202502 250
종로구 202503 190
종로구 202504 176
종로구 202505 181
종로구 202506 210
종로구 202507 203
종로구 202508 194
종로구 202509 217
종로구 202510 197
종로구 202511 221
종로구 202512 290
중구 202301 304
중구 202302 434
중구 202303 386
중구 202304 410
중구 202305 576
중구 202306 494
중구 202307 368
중구 202308 341
중구 202309 363
중구 202310 332
중구 202311 312
중구 202312 319
중구 202501 311
중구 202502 375
중구 202503 409
중구 202504 373
중구 202505 430
중구 202506 435
중구 202507 366
중구 202508 315
중구 202509 368
중구 202510 286
중구 202511 311
중구 202512 375
용산구 202301 675
용산구 202302 559
용산구 202303 639
용산구 202304 598
용산구 202305 566
용산구 202306 642
용산구 202307 574
용산구 202308 507
용산구 202309 505
용산구 202310 466
용산구 202311 465
용산구 202312 545
용산구 202501 442
용산구 202502 571
용산구 202503 596
용산구 202504 596
용산구 202505 596
용산구 202506 609
용산구 202507 580
용산구 202508 453
용산구 

In [49]:
df_seoul_3years = pd.concat(
    [df_seoul_2024, df_seoul_extra],
    ignore_index=True
)

print("3개년 전체:", df_seoul_3years.shape)

3개년 전체: (812501, 26)


In [50]:
print("전체 행:", len(df_seoul_3years))
print("완전 중복 행:", df_seoul_3years.duplicated().sum())

전체 행: 812501
완전 중복 행: 36530


In [51]:
df_seoul_3years = df_seoul_3years.drop_duplicates().reset_index(drop=True)

print("중복 제거 후:", df_seoul_3years.shape)
print("남은 중복:", df_seoul_3years.duplicated().sum())

중복 제거 후: (775971, 26)
남은 중복: 0


In [52]:
df_seoul_3years["dealYear"] = pd.to_numeric(
    df_seoul_3years["dealYear"], errors="coerce"
)

print(df_seoul_3years["dealYear"].value_counts().sort_index())

dealYear
2023    273194
2024    240326
2025    262451
Name: count, dtype: int64


In [53]:
dup_2024 = df_seoul_2024[df_seoul_2024.duplicated(keep=False)]

print("2024 중복 관련 행:", len(dup_2024))
print(dup_2024.head())

2024 중복 관련 행: 24230
       aptNm      aptSeq buildYear contractTerm contractType dealDay  \
23   인왕산아이파크  11110-2212      2008  24.02~26.02           신규      10   
32  힐스테이트창경궁  11110-2584      2022  24.03~26.03           신규      14   
38        현대    11110-90      2000  24.02~26.02           신규      12   
42       아남1    11110-25      1995  24.02~26.02           신규       2   
72  힐스테이트창경궁  11110-2584      2022  24.02~26.02           신규      16   

   dealMonth dealYear deposit excluUseAr  ... roadnmbcd roadnmbonbun  \
23         1     2024  80,000     84.858  ...         0        00009   
32         1     2024  90,000      84.95  ...         0        00236   
38         1     2024  48,500         60  ...         0        00246   
42         1     2024  50,000       84.9  ...         0        00265   
72         1     2024  95,000      84.97  ...         0        00236   

   roadnmbubun roadnmcd roadnmseq roadnmsggcd  sggCd umdNm useRRRight guName  
23       00000  4100482         1  

In [54]:
df_seoul_3years = pd.concat(
    [df_seoul_2024, df_seoul_extra],
    ignore_index=True
)

print(df_seoul_3years.shape)
print(df_seoul_3years["dealYear"].value_counts().sort_index())

(812501, 26)
dealYear
2023    284380
2024    252914
2025    275207
Name: count, dtype: int64


In [55]:
df_clean = df_seoul_3years.copy()

df_clean["deposit"] = (
    df_clean["deposit"]
    .str.replace(",", "", regex=False)
    .astype(int)
)

df_clean["monthlyRent"] = pd.to_numeric(
    df_clean["monthlyRent"], errors="coerce"
)

numeric_cols = [
    "excluUseAr", "floor", "buildYear",
    "dealYear", "dealMonth", "dealDay"
]

for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

df_clean["dealDate"] = pd.to_datetime(
    df_clean["dealYear"].astype(str) + "-" +
    df_clean["dealMonth"].astype(str) + "-" +
    df_clean["dealDay"].astype(str)
)

df_clean["rentType"] = df_clean["monthlyRent"].apply(
    lambda x: "전세" if x == 0 else "월세"
)

print(df_clean.shape)
print(df_clean["rentType"].value_counts())

(812501, 28)
rentType
전세    464333
월세    348168
Name: count, dtype: int64


In [56]:
df_clean.to_csv(
    "../data/processed/seoul_rent_2023_2025_clean.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", df_clean.shape)

저장 완료: (812501, 28)


In [57]:
df_clean.to_sql(
    "rental_transactions",
    engine,
    if_exists="replace",
    index=False,
    chunksize=5000
)

print("MySQL 저장 완료")

MySQL 저장 완료


In [58]:
monthly_check = pd.read_sql("""
SELECT
    guName,
    CONCAT(dealYear, '-', LPAD(dealMonth, 2, '0')) AS yearMonth,
    COUNT(*) AS contractCount
FROM rental_transactions
GROUP BY guName, dealYear, dealMonth
ORDER BY guName, dealYear, dealMonth
""", engine)

print(monthly_check.shape)
print(monthly_check["yearMonth"].min())
print(monthly_check["yearMonth"].max())

(900, 3)
2023-01
2025-12


In [62]:
kosis = pd.read_csv(
    "../data/raw/kosis_migration_2023_2025.csv",
    encoding="utf-8",
    header=[0, 1]
)

print(kosis.shape)
print(kosis.columns[:10])

(25, 289)
MultiIndex([('행정구역(시군구)별',        '행정구역(시군구)별'),
            (   '2023.01',           '총전입 (명)'),
            (   '2023.01',           '총전출 (명)'),
            (   '2023.01',           '순이동 (명)'),
            (   '2023.01',    '시도내이동-시군구내 (명)'),
            (   '2023.01', '시도내이동-시군구간 전입 (명)'),
            (   '2023.01', '시도내이동-시군구간 전출 (명)'),
            (   '2023.01',         '시도간전입 (명)'),
            (   '2023.01',         '시도간전출 (명)'),
            (   '2023.02',           '총전입 (명)')],
           )


In [63]:
rows = []

for _, row in kosis.iterrows():
    gu = row.iloc[0]

    for year in [2023, 2024, 2025]:
        for month in range(1, 13):
            ym = f"{year}.{month:02d}"

            rows.append({
                "guName": gu,
                "yearMonth": ym.replace(".", "-"),
                "moveIn": row[(ym, "총전입 (명)")],
                "moveOut": row[(ym, "총전출 (명)")],
                "netMove": row[(ym, "순이동 (명)")]
            })

migration = pd.DataFrame(rows)

for col in ["moveIn", "moveOut", "netMove"]:
    migration[col] = pd.to_numeric(migration[col], errors="coerce")

print(migration.shape)
print(migration.head())

(900, 5)
  guName yearMonth  moveIn  moveOut  netMove
0    종로구   2023-01    1420     1568     -148
1    종로구   2023-02    2298     2062      236
2    종로구   2023-03    1792     2124     -332
3    종로구   2023-04    1255     1408     -153
4    종로구   2023-05    1460     1698     -238


In [64]:
migration.to_sql(
    "migration",
    engine,
    if_exists="replace",
    index=False
)

print("MySQL migration:", len(migration))

MySQL migration: 900


In [65]:
ecos = pd.read_csv(
    "../data/raw/ecos_base_rate_2023_2025.csv",
    encoding="utf-8"
)

print(ecos.shape)
print(ecos.columns)

(1, 40)
Index(['통계표', '계정항목', '단위', '변환', '2023/01', '2023/02', '2023/03', '2023/04',
       '2023/05', '2023/06', '2023/07', '2023/08', '2023/09', '2023/10',
       '2023/11', '2023/12', '2024/01', '2024/02', '2024/03', '2024/04',
       '2024/05', '2024/06', '2024/07', '2024/08', '2024/09', '2024/10',
       '2024/11', '2024/12', '2025/01', '2025/02', '2025/03', '2025/04',
       '2025/05', '2025/06', '2025/07', '2025/08', '2025/09', '2025/10',
       '2025/11', '2025/12'],
      dtype='str')


In [66]:
rate_row = ecos.iloc[0]

rates = []

for year in [2023, 2024, 2025]:
    for month in range(1, 13):
        col = f"{year}/{month:02d}"

        rates.append({
            "yearMonth": f"{year}-{month:02d}",
            "baseRate": rate_row[col]
        })

interest_rate = pd.DataFrame(rates)
interest_rate["baseRate"] = pd.to_numeric(
    interest_rate["baseRate"], errors="coerce"
)

print(interest_rate.shape)
print(interest_rate.head())
print(interest_rate.tail())

(36, 2)
  yearMonth  baseRate
0   2023-01       3.5
1   2023-02       3.5
2   2023-03       3.5
3   2023-04       3.5
4   2023-05       3.5
   yearMonth  baseRate
31   2025-08       2.5
32   2025-09       2.5
33   2025-10       2.5
34   2025-11       2.5
35   2025-12       2.5


In [67]:
interest_rate.to_sql(
    "interest_rate",
    engine,
    if_exists="replace",
    index=False
)

print("MySQL interest_rate:", len(interest_rate))

MySQL interest_rate: 36


In [69]:
pop_2023 = pd.read_csv("../data/raw/LOCAL_PEOPLE_GU_2023.csv", encoding="cp949")
pop_2024 = pd.read_csv("../data/raw/LOCAL_PEOPLE_GU_2024.csv", encoding="utf-8")
pop_2025 = pd.read_csv("../data/raw/LOCAL_PEOPLE_GU_2025.csv", encoding="cp949")

seoul_pop = pd.concat(
    [pop_2023, pop_2024, pop_2025],
    ignore_index=True
)

print(seoul_pop.shape)
print(seoul_pop["stdr_de_id"].min())
print(seoul_pop["stdr_de_id"].max())

(656400, 65)
20240101.0
20251231.0


In [70]:
print(pop_2023.shape)
print(pop_2023.columns.tolist())
print(pop_2023.head(2))

(219000, 32)
['기준일ID', '시간대구분', '자치구코드', '총생활인구수', '남자0세부터9세생활인구수', '남자10세부터14세생활인구수', '남자15세부터19세생활인구수', '남자20세부터24세생활인구수', '남자25세부터29세생활인구수', '남자30세부터34세생활인구수', '남자35세부터39세생활인구수', '남자40세부터44세생활인구수', '남자45세부터49세생활인구수', '남자50세부터54세생활인구수', '남자55세부터59세생활인구수', '남자60세부터64세생활인구수', '남자65세부터69세생활인구수', '남자70세이상생활인구수', '여자0세부터9세생활인구수', '여자10세부터14세생활인구수', '여자15세부터19세생활인구수', '여자20세부터24세생활인구수', '여자25세부터29세생활인구수', '여자30세부터34세생활인구수', '여자35세부터39세생활인구수', '여자40세부터44세생활인구수', '여자45세부터49세생활인구수', '여자50세부터54세생활인구수', '여자55세부터59세생활인구수', '여자60세부터64세생활인구수', '여자65세부터69세생활인구수', '여자70세이상생활인구수']
      기준일ID  시간대구분  자치구코드       총생활인구수  남자0세부터9세생활인구수  남자10세부터14세생활인구수  \
0  20230101      0  11110  211983.7554      5256.3035        2985.2135   
1  20230101      0  11140  161323.9236      5093.4474        1916.9721   

   남자15세부터19세생활인구수  남자20세부터24세생활인구수  남자25세부터29세생활인구수  남자30세부터34세생활인구수  ...  \
0        5585.3388       10029.3603       10311.7458         8131.259  ...   
1        2969.1968        6355.1874        7354.

In [71]:
print(pop_2025.shape)
print(pop_2025.columns[:5].tolist())

(217800, 33)
['stdr_de_id', 'tmzon_pd_se', 'adstrd_code_se', 'tot_lvpop_co', 'male_f0t9_lvpop_co']


In [72]:
pop_2023 = pop_2023.rename(columns={
    "기준일ID": "stdr_de_id",
    "자치구코드": "adstrd_code_se",
    "총생활인구수": "tot_lvpop_co"
})

pop_all = pd.concat([
    pop_2023[["stdr_de_id", "adstrd_code_se", "tot_lvpop_co"]],
    pop_2024[["stdr_de_id", "adstrd_code_se", "tot_lvpop_co"]],
    pop_2025[["stdr_de_id", "adstrd_code_se", "tot_lvpop_co"]]
], ignore_index=True)

print(pop_all.shape)
print(pop_all["stdr_de_id"].min())
print(pop_all["stdr_de_id"].max())

(656400, 3)
20230101
20251231


In [73]:
pop_all["yearMonth"] = (
    pop_all["stdr_de_id"].astype(str).str[:4]
    + "-"
    + pop_all["stdr_de_id"].astype(str).str[4:6]
)

code_to_gu = {int(code): name for name, code in seoul_gu.items()}

pop_all["guName"] = pop_all["adstrd_code_se"].map(code_to_gu)

living_pop = (
    pop_all
    .groupby(["guName", "yearMonth"])["tot_lvpop_co"]
    .mean()
    .reset_index()
    .rename(columns={"tot_lvpop_co": "avgLivingPop"})
)

print(living_pop.shape)
print(living_pop.head())

(900, 3)
  guName yearMonth   avgLivingPop
0    강남구   2023-01  796674.223998
1    강남구   2023-02  831518.858496
2    강남구   2023-03  823731.966718
3    강남구   2023-04  809914.335772
4    강남구   2023-05  807608.549482


In [74]:
living_pop.to_sql(
    "living_population",
    engine,
    if_exists="replace",
    index=False
)

print("MySQL living_population:", len(living_pop))

MySQL living_population: 900


In [75]:
ml_data = pd.read_sql("""
SELECT
    r.guName,
    r.yearMonth,
    r.contractCount,
    m.moveIn,
    m.moveOut,
    m.netMove,
    i.baseRate,
    l.avgLivingPop
FROM (
    SELECT
        guName,
        CONCAT(dealYear, '-', LPAD(dealMonth, 2, '0')) AS yearMonth,
        COUNT(*) AS contractCount
    FROM rental_transactions
    GROUP BY guName, dealYear, dealMonth
) r
JOIN migration m
    ON r.guName = m.guName
    AND r.yearMonth = m.yearMonth
JOIN interest_rate i
    ON r.yearMonth = i.yearMonth
JOIN living_population l
    ON r.guName = l.guName
    AND r.yearMonth = l.yearMonth
ORDER BY r.guName, r.yearMonth
""", engine)

print(ml_data.shape)
print(ml_data["yearMonth"].min())
print(ml_data["yearMonth"].max())
print(ml_data.isnull().sum())

(900, 8)
2023-01
2025-12
guName           0
yearMonth        0
contractCount    0
moveIn           0
moveOut          0
netMove          0
baseRate         0
avgLivingPop     0
dtype: int64


In [76]:
ml_data.to_sql(
    "ml_rent_market",
    engine,
    if_exists="replace",
    index=False
)

print("MySQL ml_rent_market:", len(ml_data))

MySQL ml_rent_market: 900
